In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import requests

In [2]:
# HJNC 원본 데이터
SOURCE = Path(
    r"/hjnc_data/HJNC_T_2025.csv"
)

# 수집 결과 저장 폴더
RESULT_DIR = Path(
    r"/hjnc_data/history"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

# 최종 결과 파일
HISTORY_CSV = RESULT_DIR / "HJNC_2025_T_history.csv"
LOG_CSV = RESULT_DIR / "HJNC_2025_T_history_log.csv"

# 100개 단위 중간 저장 폴더
BATCH_DIR = RESULT_DIR / "HJNC_2025_T_history_batches"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

# HJNC API 주소
INFO_URL = "https://www.hjnc.co.kr/esvc/cntr/info/data"

HISTORY_URL = (
    "https://www.hjnc.co.kr/esvc/cntr/info/statusHistory/data"
)

# 수집 설정
SAVE_INTERVAL = 100
REQUEST_INTERVAL = 3.0
MAX_RETRIES = 3

# 이번 실행에서 처리할 컨테이너 수
TEST_COUNT = 5

session = requests.Session()
last_request_time = 0.0

print("원본 파일 존재:", SOURCE.exists())
print("결과 저장 폴더:", RESULT_DIR)

원본 파일 존재: True
결과 저장 폴더: C:\Users\user\Desktop\data_practice\busan_project\hjnc_data\history


In [3]:
# 원본 CSV 읽기
containNum = pd.read_csv(
    SOURCE,
    encoding="utf-8-sig",
    dtype="string"
)

# 필요한 3개 컬럼만 추출
target_df = containNum[
    ["컨테이너번호", "모선항차", "Sz/Tp"]
].copy()

# 문자열 앞뒤 공백 제거
for col in target_df.columns:
    target_df[col] = target_df[col].str.strip()

# 빈 값 제거
target_df = target_df.dropna()

target_df = target_df[
    (target_df["컨테이너번호"] != "") &
    (target_df["모선항차"] != "") &
    (target_df["Sz/Tp"] != "")
]

# 동일 컨테이너 + 모선항차 중복 제거
target_df = target_df.drop_duplicates(
    subset=["컨테이너번호", "모선항차"]
).reset_index(drop=True)

total = len(target_df)

print("전체 수집 대상:", total)

display(target_df.head(10))

전체 수집 대상: 47071


,컨테이너번호,모선항차,Sz/Tp
0,AFFU4007553,YWDM/0002,4500
1,AMFU8905690,YWDM/0002,4500
2,BEAU4195166,YWDM/0002,4500
3,BEAU4318297,YWDM/0002,4500
4,BEAU4329136,YWDM/0002,4500
5,BEAU4413041,HOHA/0003,4500
6,BEAU4925602,YWDM/0002,4500
7,BEAU5495770,HOHA/0003,4500
8,BEAU5517210,HOHA/0003,4500
9,BMOU4658500,HOHA/0003,4500


In [4]:
def request_json(url, params):

    global last_request_time

    for attempt in range(1, MAX_RETRIES + 1):

        # 요청 간 최소 간격 유지
        elapsed = time.monotonic() - last_request_time

        if elapsed < REQUEST_INTERVAL:
            time.sleep(REQUEST_INTERVAL - elapsed)

        try:
            last_request_time = time.monotonic()

            response = session.get(
                url,
                params=params,
                timeout=30
            )

            # HTTP 오류 확인
            response.raise_for_status()

            # JSON 변환
            return response.json()

        except (
            requests.ConnectionError,
            requests.Timeout,
            requests.HTTPError
        ) as e:

            # 재시도 가능한 오류인지 판단
            status = (
                e.response.status_code
                if isinstance(e, requests.HTTPError)
                and e.response is not None
                else None
            )

            retryable = (
                isinstance(e, (
                    requests.ConnectionError,
                    requests.Timeout
                ))
                or (status is not None and status >= 500)
            )

            # 401, 403, 429 등은 즉시 중단
            if not retryable:
                raise

            if attempt == MAX_RETRIES:
                raise

            wait = 5 * (2 ** (attempt - 1))

            print(
                f"요청 오류: {wait}초 후 재시도 "
                f"({attempt}/{MAX_RETRIES})"
            )

            time.sleep(wait)

In [5]:
def get_container_history(container_no, csv_voyage, sztp):

    # 1. 컨테이너 기본정보 조회
    data = request_json(
        INFO_URL,
        {
            "cntrNo": container_no,
            "CNTR_UID": ""
        }
    )

    voyage_list = data.get("vesselVoyageList") or []

    matched = []

    # 2. CSV 모선항차와 API 모선항차 비교
    for voyage in voyage_list:

        api_voyage = voyage.get("vesselVoyage", "")

        parts = api_voyage.split("-")

        # 예: YWDM-2024-0002 -> YWDM/0002
        if len(parts) == 3:

            converted = f"{parts[0]}/{parts[2]}"

            if converted == csv_voyage:
                matched.append(voyage)

    # 3. 모선항차가 일치하지 않는 경우
    if len(matched) == 0:
        return [], "항차 불일치", ""

    # 4. 여러 항차가 중복 매칭된 경우
    if len(matched) > 1:

        candidates = ",".join(
            v.get("vesselVoyage", "")
            for v in matched
        )

        return [], "항차 중복", candidates

    # 5. 일치하는 컨테이너 이력 선택
    selected = matched[0]

    cntr_id = selected["cntrId"]
    api_voyage = selected["vesselVoyage"]

    # 6. 상태 History 조회
    history_data = request_json(
        HISTORY_URL,
        {
            "cntrId": cntr_id,
            "sztp2": sztp
        }
    )

    if not isinstance(history_data, list):
        raise ValueError("HISTORY 응답 형식 오류")

    # 7. 작업 이력에 컨테이너 식별정보 추가
    for row in history_data:

        row["컨테이너번호"] = container_no
        row["CSV_모선항차"] = csv_voyage
        row["API_모선항차"] = api_voyage
        row["CNTR_ID"] = cntr_id

    if not history_data:
        return [], "HISTORY 0건", api_voyage

    return history_data, "정상 수집", api_voyage

In [6]:
HISTORY_COLS = [
    "ACTUAL_TIME",
    "YT_NO",
    "YJOB_TYPE",
    "OLD_POS",
    "EQU_NO",
    "NEW_POS",
    "컨테이너번호",
    "CSV_모선항차",
    "API_모선항차",
    "CNTR_ID"
]

LOG_COLS = [
    "컨테이너번호",
    "CSV_모선항차",
    "API_모선항차",
    "상태",
    "HISTORY건수",
    "오류내용"
]


def load_completed_keys():

    completed = set()

    # 이전에 저장된 로그 파일 확인
    for log_path in sorted(
        BATCH_DIR.glob("batch_*_log.csv")
    ):

        batch_log = pd.read_csv(
            log_path,
            encoding="utf-8-sig",
            dtype="string"
        )

        valid = batch_log[
            batch_log["상태"] != "API 오류"
        ]

        completed.update(
            zip(
                valid["컨테이너번호"],
                valid["CSV_모선항차"]
            )
        )

    return completed


completed_keys = load_completed_keys()

print("이미 처리 완료:", len(completed_keys))
print("전체 수집 대상:", total)
print("남은 수집 대상:", total - len(completed_keys))

이미 처리 완료: 43151
전체 수집 대상: 47071
남은 수집 대상: 3920


In [7]:
history_buffer = []
log_buffer = []

batch_number = 0


def save_batch():

    global batch_number

    if not log_buffer:
        return

    batch_number += 1

    # 기존 배치 파일과 이름이 겹치지 않도록 확인
    while (
        (
            BATCH_DIR /
            f"batch_{batch_number:08d}_log.csv"
        ).exists()
        or
        (
            BATCH_DIR /
            f"batch_{batch_number:08d}_history.csv"
        ).exists()
    ):
        batch_number += 1

    history_path = (
        BATCH_DIR /
        f"batch_{batch_number:08d}_history.csv"
    )

    log_path = (
        BATCH_DIR /
        f"batch_{batch_number:08d}_log.csv"
    )

    history_temp = history_path.with_suffix(".csv.tmp")
    log_temp = log_path.with_suffix(".csv.tmp")

    # 작업 이력 데이터프레임
    history_df = pd.DataFrame(
        history_buffer
    ).reindex(columns=HISTORY_COLS)

    # 조회 로그 데이터프레임
    log_df = pd.DataFrame(
        log_buffer
    ).reindex(columns=LOG_COLS)

    # 임시 파일에 먼저 저장
    history_df.to_csv(
        history_temp,
        index=False,
        encoding="utf-8-sig"
    )

    log_df.to_csv(
        log_temp,
        index=False,
        encoding="utf-8-sig"
    )

    # 정식 CSV 파일로 확정
    os.replace(history_temp, history_path)
    os.replace(log_temp, log_path)

    print(
        f"배치 저장 완료: "
        f"대상 {len(log_df)}개 / "
        f"HISTORY {len(history_df)}건"
    )

    history_buffer.clear()
    log_buffer.clear()

In [10]:
# ==========================================
# 8. 전체 컨테이너 HISTORY 수집
# ==========================================

start_time = time.time()
processed = 0

# 기존 수집 완료 목록 불러오기
completed_keys = load_completed_keys()

# 이미 수집한 컨테이너는 제외하고 전체 조회
pending_df = target_df[
    [
        (cn, voyage) not in completed_keys
        for cn, voyage in zip(
            target_df["컨테이너번호"],
            target_df["모선항차"]
        )
    ]
].copy()

print("전체 수집 대상:", len(target_df))
print("기존 처리 완료:", len(completed_keys))
print("이번 실행 대상:", len(pending_df))
print("\n전체 HISTORY 수집 시작\n")

try:
    for i, (container_no, csv_voyage, sztp) in enumerate(
        pending_df.itertuples(index=False, name=None),
        start=1
    ):

        print(f"\n[{i}/{len(pending_df)}] {container_no}")

        try:
            # 컨테이너 HISTORY 조회
            rows, status, api_voyage = get_container_history(
                container_no,
                csv_voyage,
                sztp
            )

            error = ""

        except Exception as e:
            rows = []
            status = "API 오류"
            api_voyage = ""
            error = str(e)

        # 작업 이력 누적
        history_buffer.extend(rows)

        # 조회 로그 누적
        log_buffer.append({
            "컨테이너번호": container_no,
            "CSV_모선항차": csv_voyage,
            "API_모선항차": api_voyage,
            "상태": status,
            "HISTORY건수": len(rows),
            "오류내용": error
        })

        processed += 1

        print(
            f"{csv_voyage} | "
            f"{status} | {len(rows)}건"
        )

        # 100개마다 중간 저장
        if processed % SAVE_INTERVAL == 0:
            save_batch()

        # API 오류 발생 시 수집 중단
        if status == "API 오류":
            print("API 오류로 수집을 중단합니다.")
            print("오류 내용:", error)
            break

except KeyboardInterrupt:
    print("\n사용자가 수집을 중단했습니다.")

finally:
    # 남은 데이터 저장
    save_batch()

    elapsed = (time.time() - start_time) / 60

    print("\n==============================")
    print("이번 실행 처리:", processed)
    print(f"소요 시간: {elapsed:.1f}분")
    print("==============================")

전체 수집 대상: 47071
기존 처리 완료: 45899
이번 실행 대상: 1172

전체 HISTORY 수집 시작


[1/1172] KKFU7933794
HOHL/0001 | 정상 수집 | 4건

[2/1172] KKFU8066759
HOHL/0001 | 정상 수집 | 4건

[3/1172] KKFU8145020
HOHL/0001 | 정상 수집 | 4건

[4/1172] KKFU8164441
HOHL/0001 | 정상 수집 | 4건

[5/1172] MOTU1418068
HOHL/0001 | 정상 수집 | 4건

[6/1172] NYKU4306467
HOHL/0001 | 정상 수집 | 4건

[7/1172] NYKU4365881
HOHL/0001 | 정상 수집 | 4건

[8/1172] NYKU4750419
HOHL/0001 | 정상 수집 | 4건

[9/1172] NYKU4774004
HOHL/0001 | 정상 수집 | 4건

[10/1172] NYKU4812357
HOHL/0001 | 정상 수집 | 4건

[11/1172] ONEU0186289
HOHL/0001 | 정상 수집 | 4건

[12/1172] ONEU0251044
HOHL/0001 | 정상 수집 | 4건

[13/1172] ONEU0297934
HOHL/0001 | 정상 수집 | 4건

[14/1172] ONEU0487235
HOHL/0001 | 정상 수집 | 4건

[15/1172] ONEU0608830
HOHL/0001 | 정상 수집 | 4건

[16/1172] ONEU0648896
HOHL/0001 | 정상 수집 | 4건

[17/1172] ONEU0798828
HOHL/0001 | 정상 수집 | 4건

[18/1172] ONEU0832558
HOHL/0001 | 정상 수집 | 4건

[19/1172] ONEU0978080
HOHL/0001 | 정상 수집 | 4건

[20/1172] ONEU1012445
HOHL/0001 | 정상 수집 | 4건

[21/1172] ONEU1032652


In [11]:
def merge_batches():

    history_paths = sorted(
        BATCH_DIR.glob("batch_*_history.csv")
    )

    log_paths = sorted(
        BATCH_DIR.glob("batch_*_log.csv")
    )

    # 1. 로그 파일 통합
    log_frames = [
        pd.read_csv(
            path,
            encoding="utf-8-sig",
            dtype="string"
        )
        for path in log_paths
    ]

    if log_frames:

        final_log = pd.concat(
            log_frames,
            ignore_index=True
        )

        final_log = final_log.drop_duplicates(
            subset=["컨테이너번호", "CSV_모선항차"],
            keep="last"
        )

        final_log.to_csv(
            LOG_CSV,
            index=False,
            encoding="utf-8-sig"
        )

        # 정상 수집된 항차만 선별
        valid_keys = set(
            zip(
                final_log.loc[
                    final_log["상태"] == "정상 수집",
                    "컨테이너번호"
                ],
                final_log.loc[
                    final_log["상태"] == "정상 수집",
                    "CSV_모선항차"
                ]
            )
        )

    else:
        valid_keys = set()

    # 2. HISTORY 파일 통합
    history_frames = [
        pd.read_csv(
            path,
            encoding="utf-8-sig",
            dtype="string"
        )
        for path in history_paths
    ]

    if history_frames:

        final_history = pd.concat(
            history_frames,
            ignore_index=True
        )

        keys = list(zip(
            final_history["컨테이너번호"],
            final_history["CSV_모선항차"]
        ))

        final_history = final_history[
            [key in valid_keys for key in keys]
        ]

        # 중복 작업 기록 제거
        final_history = final_history.drop_duplicates(
            subset=[
                "컨테이너번호",
                "CSV_모선항차",
                "CNTR_ID",
                "ACTUAL_TIME",
                "YJOB_TYPE",
                "OLD_POS",
                "NEW_POS",
                "EQU_NO",
                "YT_NO"
            ]
        )

    else:
        final_history = pd.DataFrame(
            columns=HISTORY_COLS
        )

    final_history.to_csv(
        HISTORY_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print("최종 CSV 통합 완료")
    print("HISTORY:", HISTORY_CSV)
    print("LOG:", LOG_CSV)
    print("전체 작업 이력:", len(final_history))

    return final_history


final_history = merge_batches()

display(final_history.head(20))

최종 CSV 통합 완료
HISTORY: C:\Users\user\Desktop\data_practice\busan_project\hjnc_data\history\HJNC_2025_T_history.csv
LOG: C:\Users\user\Desktop\data_practice\busan_project\hjnc_data\history\HJNC_2025_T_history_log.csv
전체 작업 이력: 213268


,ACTUAL_TIME,YT_NO,YJOB_TYPE,OLD_POS,EQU_NO,NEW_POS,컨테이너번호,CSV_모선항차,API_모선항차,CNTR_ID
0,2025-01-01 12:07:48,YT501,2Time Shifting Loading,<NA>,GC106,86D-10-70,AFFU4007553,YWDM/0002,YWDM-2024-0002,0000042995989
1,2025-01-01 12:04:46,YT501,Lift On For Loading,---,TC223,2B-44-4-6,AFFU4007553,YWDM/0002,YWDM-2024-0002,0000042995989
2,2025-01-01 10:06:00,YT544,Lift Off for Discharging,---,TC223,2B-44-4-6,AFFU4007553,YWDM/0002,YWDM-2024-0002,0000042995989
3,2025-01-01 10:01:03,YT544,2Time Shifting Discharging,66D-13-72,GC104,<NA>,AFFU4007553,YWDM/0002,YWDM-2024-0002,0000042995989
4,2025-01-01 11:06:02,YT542,2Time Shifting Loading,<NA>,GC101,26D-07-72,AMFU8905690,YWDM/0002,YWDM-2024-0002,0000042995067
5,2025-01-01 10:52:46,YT542,Lift On For Loading,---,TC224,2B-24-3-4,AMFU8905690,YWDM/0002,YWDM-2024-0002,0000042995067
6,2025-01-01 04:55:11,<NA>,Auto Shifting,2B-24-3-4,TC223,2B-24-7-6,AMFU8905690,YWDM/0002,YWDM-2024-0002,0000042995067
7,2024-12-31 22:46:02,YT552,Lift Off for Discharging,---,TC223,2B-24-7-6,AMFU8905690,YWDM/0002,YWDM-2024-0002,0000042995067
8,2024-12-31 22:41:22,YT552,2Time Shifting Discharging,38D-15-78,GC102,<NA>,AMFU8905690,YWDM/0002,YWDM-2024-0002,0000042995067
9,2025-01-01 11:13:37,YT539,2Time Shifting Loading,<NA>,GC101,26D-07-74,BEAU4195166,YWDM/0002,YWDM-2024-0002,0000042994961
